# Project 08: HMM - The Viterbi Algorithm

## Provided Data Structures

In [2]:
import numpy as np
class HMM:
    """Class for HMM

        Attributes:
        init (dictionary): initial probabilities (probability of starting in each state) 
        trans (dictionary): transition probabilities (probability of moving from one state to another)
        emit (dictionary): emission probabilities (probability of observing a symbol in a given state)
        states (list): list of states
    """
    def __init__(self, init_probs, trans_probs, emit_probs):
        """Initialize the object.
        
        Args:
            init_pobs (dictionary): initial probabilities (probability of starting in each state) 
            trans_probs (dictionary): transition probabilities (probability of moving from one state to another)
            emit_probs (dictionary): emission probabilities (probability of observing a symbol in a given state)
        """
        self.init = init_probs
        self.trans = trans_probs
        self.emit = emit_probs
        self.states = list(emit_probs.keys())

    def viterbi(self, sequence):
        """Runs Viterbi algorithm for given sequence

        Args:
            sequence (string): DNA sequence to analyze

        Returns:
        log_probability_matrix (numpy matrix): observations x state (max log probability of that observation given previous log probabilities and state)
        traceback_matrix (numpy matrix): observations x state (row (state) that the max probability for that observation came from)
        """

        # create traceback matrix with 0's with correct size
        traceback_matrix = np.zeros([len(self.states), len(sequence)], dtype=int)

        # create probability matrix of 0's with correct size
        log_probability_matrix =  np.zeros((len(self.states), len(sequence)))

        # initialize first column using initial probabilities
        for row_num in range(len(self.states)):
            state = self.states[row_num]         # get state as its letter key
            log_probability_matrix[row_num, 0] = np.log(self.init[state] * self.emit[state][sequence[0]])

        # iterate through remaining columns, finding max probability at each row position
        for column_num in (range(1, len(sequence))):
            current_nucletotide = sequence[column_num]    # what is the observation
            
            # for each row in this coulumn
            for row_num in range(len(self.states)):
                current_state = self.states[row_num]    # what is the state of this row
                last_column = column_num - 1        # what is the last column (to get last probabilities)
                log_prob_dictionary = {}        
                
                # for each row (probability) in the last the column, calculate the probability of it producing the current state and observation, then take the max 
                for last_row_num in range(len(self.states)):
                    last_state = self.states[last_row_num]      # get last state as letter key
                    log_last_state_prob = log_probability_matrix[last_row_num, last_column]     # get probabilities from last column
                    log_prob =  log_last_state_prob + np.log(self.trans[last_state][current_state] * self.emit[current_state][current_nucletotide]) # calculate probabilites for current
                    log_prob_dictionary[last_state] = log_prob      # store probabilites in dictionary by last state that produced them   

                max_state = max(log_prob_dictionary, key=log_prob_dictionary.get)       # get last state (key) that produced the max probability 
                log_probability_matrix[row_num, column_num] = log_prob_dictionary[max_state]   
                traceback_matrix[row_num, column_num] = self.states.index(max_state)        # store the row (index) that produced the max probability

                                
        # Termination
        # Last State
        best_last_row = np.argmax(log_probability_matrix[:, -1])        
        best_prob = log_probability_matrix[best_last_row, -1]

        # Traceback (row = state)
        path_rows = [best_last_row]     # assign state at end to the highest probability at end 
        current_row = best_last_row
        for column_num in range(len(sequence) - 1, 0, -1):   # last column down to 1
            previous_row = traceback_matrix[current_row, column_num]        # use traceback matrix to determine previous states
            path_rows.append(previous_row)
            current_row = previous_row

        path_rows.reverse()
        path = [self.states[i] for i in path_rows]

        return path, best_prob, log_probability_matrix, traceback_matrix 
    
    


In [6]:


# Example observation sequence
obs = "GGCACTGAA"
obs2 = "ATATATCGCGCCGCGCGCATATAT"

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "I": 0.2,
    "G": 0.8
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "I": {"I": 0.7, "G": 0.3},
    "G": {"I": 0.1, "G": 0.9}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}
}

my_hmm = HMM(init_probs, trans_probs, emit_probs)


path, best_prob, probability_matrix, traceback_matrix = my_hmm.viterbi(obs)


path2, best_prob2, probability_matrix2, traceback_matrix2 = my_hmm.viterbi(obs2)

print(path)
print(path2)


['G', 'G', 'G', 'G', 'G', 'G', 'G', 'G', 'G']
['G', 'G', 'G', 'G', 'G', 'G', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'G', 'G', 'G', 'G', 'G', 'G']
